In [0]:
T_trans=spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_item = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_item")
T_date = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_date")
T_location = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")

In [0]:
from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

start_date = '2026-03-18'
end_date = '2026-05-21'


In [0]:
tags = [
    'BP TAG+$10 - 52 CR',
    'BP TAG+$100 - 802 CR',
    'BP TAG+$129 - 1107 CR',
    'BP TAG+$15 - 72 CR',
    'BP TAG+$25 - 172 CR',
    'BP TAG+$35 - 257 CR',
    'BP TAG+$50 - 377 CR',
    'BP TAG+$75 - 577 CR',
    'PREM TAG+$10 - 54 CR',
    'PREM TAG+$100 - 804 CR',
    'PREM TAG+$129 - 1109 CR',
    'PREM TAG+$15 - 74 CR',
    'PREM TAG+$25 - 174 CR',
    'PREM TAG+$35 - 257 CR',
    'PREM TAG+$35 - 259 CR',
    'PREM TAG+$50 - 379 CR',
    'PREM TAG+$75 - 579 CR',
    'PROMO BP TAG $100 - 1002 CR',
    'TAG $10 - 40 CR',
    'TAG $100 - 790 CR',
    'TAG $129 - 1095 CR',
    'TAG $15 - 60 CR',
    'TAG $25 - 160 CR',
    'TAG $35 - 245 CR',
    'TAG $5 - 20 CR',
    'TAG $50 - 365 CR',
    'TAG $75 - 565 CR',
    'HSH TAG 60 CR'
]

tag_bins = {
    '$5': [
        'TAG $5 - 20 CR',
    ],
    '$10': [
        'BP TAG+$10 - 52 CR',
        'PREM TAG+$10 - 54 CR',
        'TAG $10 - 40 CR',
    ],
    '$15': [
        'BP TAG+$15 - 72 CR',
        'PREM TAG+$15 - 74 CR',
        'TAG $15 - 60 CR',
    ],
    '$25': [
        'BP TAG+$25 - 172 CR',
        'PREM TAG+$25 - 174 CR',
        'TAG $25 - 160 CR',
    ],
    '$35': [
        'BP TAG+$35 - 257 CR',
        'PREM TAG+$35 - 257 CR',
        'PREM TAG+$35 - 259 CR',
        'TAG $35 - 245 CR',
    ],
    '$50': [
        'BP TAG+$50 - 377 CR',
        'PREM TAG+$50 - 379 CR',
        'TAG $50 - 365 CR',
    ],
    '$75': [
        'BP TAG+$75 - 577 CR',
        'PREM TAG+$75 - 579 CR',
        'TAG $75 - 565 CR',
    ],
    '$100': [
        'BP TAG+$100 - 802 CR',
        'PREM TAG+$100 - 804 CR',
        'PROMO BP TAG $100 - 1002 CR',
        'TAG $100 - 790 CR',
    ],
    '$129': [
        'BP TAG+$129 - 1107 CR',
        'PREM TAG+$129 - 1109 CR',
        'TAG $129 - 1095 CR',
    ],
    'HSH': ['HSH TAG 60 CR']
}

In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Row

# Create mapping DataFrame
mapping_data = [
    Row(Item_Description=item, price_bin=bin_name) 
    for bin_name, items in tag_bins.items() 
    for item in items
]

bin_mapping_df = spark.createDataFrame(mapping_data)

T_game_band = (
    T_trans
    .join(
        T_date.select("dateid", "datefull"),
        T_date.dateid == T_trans.VisitDateId,
        "left"
    )
    .join(
        T_location.select(
            "LocationID",
            "Province_CD",
            "LC_Trade_Brand_Description",
            "LC_Location_Type_Description",
            "Location_Name"
        ),
        T_location.LocationID == T_trans.LocationId,
        "left"
    )
    .join(
        T_item.select(
            "itemskey",
            "Item_Description",
            "Item_Class_Description1"
        ),
        T_item.itemskey == T_trans.ItemSkey,
        "left"
    )
    .join(bin_mapping_df, on="Item_Description", how="left")

    # Add FnB Classification
    .withColumn(
        "price_bin",
        F.when(
            F.col("Item_Class_Description1").rlike("(?i)TRR - Menu"),
            "FnB"
        ).otherwise(F.col("price_bin"))
    )

    .filter(F.col("datefull").between(start_date, end_date))
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
    .filter(
        (F.col("Location_Name").like("%Granville%")) |
        (F.col("Location_Name").like("%Royal%"))
    )

    # Keep either TAG items or FnB items
    .filter(
        (F.col("Item_Description").isin(tags)) |
        (F.col("Item_Class_Description1").rlike("(?i)TRR - MENU"))
    )
)

In [0]:
T_all_game_band_summary = (
    T_game_band

    # Create week column
    .withColumn(
        "week_start",
        F.date_trunc("week", F.col("datefull"))
    )

    .groupBy("price_bin", "Location_Name", "week_start")

    .agg(
        # Total Revenue (All)
        F.sum("Net_Revenue").alias("Total_Revenue"),

        # FnB Revenue
        F.sum(
            F.when(F.col("price_bin") == "FnB", F.col("Net_Revenue"))
             .otherwise(0)
        ).alias("FnB_Revenue"),

        # Gameband Revenue (excluding FnB)
        F.sum(
            F.when(F.col("price_bin") != "FnB", F.col("Net_Revenue"))
             .otherwise(0)
        ).alias("Gameband_Revenue"),

        F.sum("Quantity").alias("Total_Sales"),

        F.sum(
            F.when(F.col("SCENEFLAG") == "Yes", 1)
             .otherwise(0)
        ).alias("Scene_Sales_Count"),

        F.sum(
            F.when(F.col("CineClubFLAG") == "Yes", 1)
             .otherwise(0)
        ).alias("CineClub_Sales_Count"),

        F.countDistinct(
            F.when(F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))
        ).alias("Distinct_Customers")
    )

    .orderBy("week_start", "price_bin")
)

In [0]:
T_all_game_band_summary.display()

In [0]:
T_pivot_revenue = (
    T_all_game_band_summary
    .groupBy("Location_Name", "price_bin")
    .pivot("week_start")
    .agg(F.sum("Total_Revenue"))
    .orderBy("Location_Name", "price_bin")
)

In [0]:
T_pivot_revenue.display()

In [0]:
T_pivot_by_bin = (
    T_all_game_band_summary
    .groupBy("Location_Name", "week_start")
    .pivot("price_bin", ["$5","$10","$15","$25","$35","$50","$75","$100","$129","HSH","FnB"])
    .agg(F.sum("Total_Revenue"))
    .orderBy("Location_Name", "week_start")
)

In [0]:
T_pivot_by_bin.display()

In [0]:
gameband_bins = ["$5", "$10", "$15", "$25", "$35", "$50", "$75", "$100", "$129", "HSH"]

T_pivot_share = (
    T_all_game_band_summary
    .filter(F.col("price_bin") != "FnB")
    .withColumn("week_start", F.col("week_start").cast("date"))
    .groupBy("Location_Name", "week_start")
    .pivot("price_bin", gameband_bins)
    .agg(F.sum("Total_Revenue"))
    .na.fill(0, subset=gameband_bins)
)

# Row total across gameband tiers
T_pivot_share = T_pivot_share.withColumn(
    "Gameband_Total",
    sum(F.col(b) for b in gameband_bins)
)

# Convert each tier to % share
for b in gameband_bins:
    T_pivot_share = T_pivot_share.withColumn(
        b,
        F.when(F.col("Gameband_Total") > 0,
               F.round(F.col(b) / F.col("Gameband_Total") * 100, 2))
         .otherwise(F.lit(0))
    )

T_pivot_share = T_pivot_share.orderBy("Location_Name", "week_start")

In [0]:
T_pivot_share.display()

In [0]:
# Get HSH's absolute revenue and total gameband revenue
hsh_stats = (
    T_all_game_band_summary
    .filter(F.col("price_bin") != "FnB")
    .groupBy("Location_Name", "week_start")
    .agg(
        F.sum(F.when(F.col("price_bin") == "HSH", F.col("Total_Revenue"))
              .otherwise(0)).alias("HSH_Rev"),
        F.sum("Total_Revenue").alias("Total_Gameband_Rev")
    )
)

# Compute weekly CV (coefficient of variation) of total gameband revenue
# This is your "noise floor"

In [0]:
hsh_stats.display()

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
from pyspark.sql import Row

# Define March Break dates by province and year
march_break_dates = {
    'ON': {'2025': '2025-03-10', '2026': '2026-03-16'},
    'AB': {'2025': '2025-03-24', '2026': '2026-03-21'},
    'BC': {'2025': '2025-03-17', '2026': '2026-03-16'},
    'MB': {'2025': '2025-03-31', '2026': '2026-03-30'},
    'QC': {'2025': '2025-03-03', '2026': '2026-04-02'},
    'NS': {'2025': '2025-03-10', '2026': '2026-03-16'},
    'NL': {'2025': '2025-04-21', '2026': '2026-03-16'},
}

# Build period conditions
conditions = []
for province, years in march_break_dates.items():
    for year, mb_start_str in years.items():
        mb_start = datetime.strptime(mb_start_str, '%Y-%m-%d')
        
        # 10 days before March Break (not including March Break start)
        conditions.append({
            'province': province,
            'year': year,
            'period': 'Pre Period (25 Days)',
            'date_start': (mb_start - timedelta(days=21)).strftime('%Y-%m-%d'),
            'date_end': (mb_start - timedelta(days=1)).strftime('%Y-%m-%d')
        })
        # 10 days into March Break (including March Break start)
        conditions.append({
            'province': province,
            'year': year,
            'period': 'Offer Period (21 Days)',
            'date_start': mb_start_str,
            'date_end': (mb_start + timedelta(days=20)).strftime('%Y-%m-%d')
        })

# Create mapping DataFrame
condition_rows = [
    Row(
        Province_CD=c['province'],
        Year=c['year'],
        Period=c['period'],
        date_start=c['date_start'],
        date_end=c['date_end']
    ) for c in conditions
]
conditions_df = spark.createDataFrame(condition_rows)

# Join and filter
T_consolidated_by_province = (
    T_all_game_band_summary
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .withColumn("Year_Period", F.concat(F.col("Year"), F.lit("_"), F.col("Period")))
    .groupBy("LC_Trade_Brand_Description")
    .pivot("Year_Period")
    .agg(
        F.sum("Total_Sales").alias("Total_Sales"),
        F.sum("Total_Revenue").alias("Total_Revenue"),
        F.sum("Scene_Sales_Count").alias("Scene_Sales_Count"),
        F.sum("CineClub_Sales_Count").alias("CineClub_Sales_Count"),
        F.sum("Distinct_Customers").alias("Distinct_Customers")
    )
    .orderBy("LC_Trade_Brand_Description")
)

T_consolidated_by_gameband = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .withColumn("Year_Period", F.concat(F.col("Year"), F.lit("_"), F.col("Period")))
    .groupBy("price_bin")
    .pivot("Year_Period")
    .agg(
        F.sum("Quantity").alias("Total_Sales"),
        F.sum("Net_Revenue").alias("Total_Revenue"),
        F.sum(F.when(F.col("SCENEFLAG") == "Yes", 1).otherwise(0)).alias("Scene_Sales_Count"),
        F.sum(F.when(F.col("CineClubFLAG") == "Yes", 1).otherwise(0)).alias("CineClub_Sales_Count"),
        F.countDistinct(F.when(F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))).alias("Distinct_Customers")
    )
    .orderBy("price_bin")
)


In [0]:
T_consolidated_by_province.display()

In [0]:
T_consolidated_by_gameband.display()

In [0]:
total_customers = customers_25_gameband_2026.count()

cross_purchases_2026 = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .filter(F.col("Period") == "Offer Period (25 Days)")
    .filter(F.col("Year") == "2026")
    .filter(F.col("CDE_ID").isNotNull())
    .join(customers_25_gameband_2026, on="CDE_ID", how="inner")
    .groupBy("price_bin")
    .agg(
        F.countDistinct("CDE_ID").alias("Distinct_Customers"),
        F.sum("Quantity").alias("Total_Quantity"),
        F.sum("Net_Revenue").alias("Total_Revenue"),
        F.try_divide(
            F.sum("Quantity"),
            F.countDistinct("CDE_ID")
        ).alias("Avg_Quantity_Per_Customer")
    )
    .withColumn("Pct_of_25_Buyers", F.round(F.col("Distinct_Customers") / F.lit(total_customers) * 100, 2))
    .orderBy("price_bin")
)

cross_purchases_2026.display()

In [0]:
# Step 1: Get distinct CDE_IDs who bought $25 gamebands in any period (with Year and Period)
customers_25_gameband = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .filter(F.col("price_bin") == "$25")
    .filter(F.col("CDE_ID").isNotNull())
    .select("CDE_ID", "Year", "Period")
    .distinct()
)

# Get total $25 buyers per Year/Period for percentage calculation
total_per_year_period = (
    customers_25_gameband
    .groupBy("Year", "Period")
    .agg(F.countDistinct("CDE_ID").alias("Total_25_Buyers"))
)

# Step 2: Find ALL gameband purchases by these customers in the SAME year and period
cross_purchases = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .filter(F.col("CDE_ID").isNotNull())
    .join(customers_25_gameband, on=["CDE_ID", "Year", "Period"], how="inner")
    .groupBy("Year", "Period", "price_bin")
    .agg(
        F.countDistinct("CDE_ID").alias("Distinct_Customers"),
        F.sum("Quantity").alias("Total_Quantity"),
        F.sum("Net_Revenue").alias("Total_Revenue"),
        F.try_divide(
            F.sum("Quantity"),
            F.countDistinct("CDE_ID")
        ).alias("Avg_Quantity_Per_Customer")
    )
    .join(total_per_year_period, on=["Year", "Period"], how="left")
    .withColumn("Pct_of_25_Buyers", F.round(F.col("Distinct_Customers") / F.col("Total_25_Buyers") * 100, 2))
    .drop("Total_25_Buyers")
    .withColumn("Year_Period", F.concat(F.col("Year"), F.lit("_"), F.col("Period")))
    .orderBy("Year", "Period", "price_bin")
)

cross_purchases.display()

In [0]:
cross_purchases_yoy = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .filter(F.col("CDE_ID").isNotNull())
    .join(customers_25_gameband, on=["CDE_ID", "Year", "Period"], how="inner")
    .groupBy("Period", "price_bin")
    .pivot("Year")
    .agg(
        F.countDistinct("CDE_ID").alias("Customers"),
        F.sum("Quantity").alias("Qty")
    )
    .withColumn("Customer_Change_Pct", 
        F.round((F.col("2026_Customers") - F.col("2025_Customers")) / F.col("2025_Customers") * 100, 2))
    .withColumn("Qty_Change_Pct",
        F.round((F.col("2026_Qty") - F.col("2025_Qty")) / F.col("2025_Qty") * 100, 2))
    .orderBy("Period", "price_bin")
)

cross_purchases_yoy.display()

In [0]:
(
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .filter(F.col("Period") == "Offer Period (25 Days)")
    .filter(F.col("Year") == "2026")
    .filter(F.col("price_bin") == "$25")
    .agg(
        F.sum("Quantity").alias("Total_25_Gamebands"),
        F.sum(F.when(F.col("CDE_ID").isNotNull(), F.col("Quantity")).otherwise(0)).alias("With_CDE_ID"),
        F.sum(F.when(F.col("CDE_ID").isNull(), F.col("Quantity")).otherwise(0)).alias("Without_CDE_ID"),
        F.sum(F.when(F.col("SCENEFLAG") == "Yes", F.col("Quantity")).otherwise(0)).alias("Scene_Sales"),
        F.sum(F.when((F.col("SCENEFLAG") == "Yes") & (F.col("CDE_ID").isNull()), F.col("Quantity")).otherwise(0)).alias("Scene_Without_CDE_ID")
    )
).display()

In [0]:
(
    T_game_band
    .filter(F.col("datefull").between("2026-03-13", "2026-04-06"))
    .filter(F.col("SCENEFLAG") == "Yes")
    .groupBy("price_bin")
    .agg(
        F.sum("Quantity").alias("Total_Scene_Gamebands"),
        F.countDistinct(F.when(F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))).alias("Distinct_Customers")
    )
    .orderBy("price_bin")
).display()

In [0]:
from pyspark.sql import functions as F
from datetime import datetime, timedelta
from pyspark.sql import Row

# Define March Break dates by province and year
march_break_dates = {
    'ON': {'2025': '2025-03-13', '2026': '2026-03-13'},
    'AB': {'2025': '2025-03-13', '2026': '2026-03-13'},
    'BC': {'2025': '2025-03-13', '2026': '2026-03-13'},
    'MB': {'2025': '2025-03-13', '2026': '2026-03-13'},
    'QC': {'2025': '2025-03-13', '2026': '2026-03-13'},
    'NS': {'2025': '2025-03-13', '2026': '2026-03-13'},
    'NL': {'2025': '2025-03-13', '2026': '2026-03-13'},
}

# Build period conditions
conditions = []
for province, years in march_break_dates.items():
    for year, mb_start_str in years.items():
        mb_start = datetime.strptime(mb_start_str, '%Y-%m-%d')
        
        # 10 days before March Break (not including March Break start)
        conditions.append({
            'province': province,
            'year': year,
            'period': 'Pre Period (25 Days)',
            'date_start': (mb_start - timedelta(days=25)).strftime('%Y-%m-%d'),
            'date_end': (mb_start - timedelta(days=1)).strftime('%Y-%m-%d')
        })
        # 10 days into March Break (including March Break start)
        conditions.append({
            'province': province,
            'year': year,
            'period': 'Offer Period (25 Days)',
            'date_start': mb_start_str,
            'date_end': (mb_start + timedelta(days=24)).strftime('%Y-%m-%d')
        })

# Create mapping DataFrame
condition_rows = [
    Row(
        Province_CD=c['province'],
        Year=c['year'],
        Period=c['period'],
        date_start=c['date_start'],
        date_end=c['date_end']
    ) for c in conditions
]
conditions_df = spark.createDataFrame(condition_rows)

# Join and filter
T_consolidated_by_province = (
    T_all_game_band_summary
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .withColumn("Year_Period", F.concat(F.col("Year"), F.lit("_"), F.col("Period")))
    .groupBy("Province_CD")
    .pivot("Year_Period")
    .agg(
        F.sum("Total_Sales").alias("Total_Sales"),
        F.sum("Total_Revenue").alias("Total_Revenue"),
        F.sum("Scene_Sales_Count").alias("Scene_Sales_Count"),
        F.sum("CineClub_Sales_Count").alias("CineClub_Sales_Count"),
        F.sum("Distinct_Customers").alias("Distinct_Customers")
    )
    .orderBy("Province_CD")
)

T_consolidated_by_gameband = (
    T_game_band
    .withColumn("Year", F.year("datefull").cast("string"))
    .join(conditions_df, on=["Province_CD", "Year"], how="inner")
    .filter(F.col("datefull").between(F.col("date_start"), F.col("date_end")))
    .withColumn("Year_Period", F.concat(F.col("Year"), F.lit("_"), F.col("Period")))
    .groupBy("price_bin")
    .pivot("Year_Period")
    .agg(
        F.sum("Quantity").alias("Total_Sales"),
        F.sum("Net_Revenue").alias("Total_Revenue"),
        F.sum(F.when(F.col("SCENEFLAG") == "Yes", 1).otherwise(0)).alias("Scene_Sales_Count"),
        F.sum(F.when(F.col("CineClubFLAG") == "Yes", 1).otherwise(0)).alias("CineClub_Sales_Count"),
        F.countDistinct(F.when(F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))).alias("Distinct_Customers")
    )
    .orderBy("price_bin")
)


In [0]:
import pyspark.sql.functions as F
from pyspark.sql import Row

# ── 1. Tag definitions ────────────────────────────────────────────────────────

tags = [
    'BP TAG+$10 - 52 CR',
    'BP TAG+$100 - 802 CR',
    'BP TAG+$129 - 1107 CR',
    'BP TAG+$15 - 72 CR',
    'BP TAG+$25 - 172 CR',
    'BP TAG+$35 - 257 CR',
    'BP TAG+$50 - 377 CR',
    'BP TAG+$75 - 577 CR',
    'PREM TAG+$10 - 54 CR',
    'PREM TAG+$100 - 804 CR',
    'PREM TAG+$129 - 1109 CR',
    'PREM TAG+$15 - 74 CR',
    'PREM TAG+$25 - 174 CR',
    'PREM TAG+$35 - 257 CR',
    'PREM TAG+$35 - 259 CR',
    'PREM TAG+$50 - 379 CR',
    'PREM TAG+$75 - 579 CR',
    'PROMO BP TAG $100 - 1002 CR',
    'TAG $10 - 40 CR',
    'TAG $100 - 790 CR',
    'TAG $129 - 1095 CR',
    'TAG $15 - 60 CR',
    'TAG $25 - 160 CR',
    'TAG $35 - 245 CR',
    'TAG $5 - 20 CR',
    'TAG $50 - 365 CR',
    'TAG $75 - 565 CR',
]

tag_bins = {
    '$5':   ['TAG $5 - 20 CR'],
    '$10':  ['BP TAG+$10 - 52 CR', 'PREM TAG+$10 - 54 CR', 'TAG $10 - 40 CR'],
    '$15':  ['BP TAG+$15 - 72 CR', 'PREM TAG+$15 - 74 CR', 'TAG $15 - 60 CR'],
    '$25':  ['BP TAG+$25 - 172 CR', 'PREM TAG+$25 - 174 CR', 'TAG $25 - 160 CR'],
    '$35':  ['BP TAG+$35 - 257 CR', 'PREM TAG+$35 - 257 CR', 'PREM TAG+$35 - 259 CR', 'TAG $35 - 245 CR'],
    '$50':  ['BP TAG+$50 - 377 CR', 'PREM TAG+$50 - 379 CR', 'TAG $50 - 365 CR'],
    '$75':  ['BP TAG+$75 - 577 CR', 'PREM TAG+$75 - 579 CR', 'TAG $75 - 565 CR'],
    '$100': ['BP TAG+$100 - 802 CR', 'PREM TAG+$100 - 804 CR', 'PROMO BP TAG $100 - 1002 CR', 'TAG $100 - 790 CR'],
    '$129': ['BP TAG+$129 - 1107 CR', 'PREM TAG+$129 - 1109 CR', 'TAG $129 - 1095 CR'],
}

# ── 2. Build price bin mapping DataFrame ──────────────────────────────────────

mapping_data = [
    Row(Item_Description=item, price_bin=bin_name)
    for bin_name, items in tag_bins.items()
    for item in items
]
bin_mapping_df = spark.createDataFrame(mapping_data)

# ── 3. Build base Gameband table (2025 only) ──────────────────────────────────

T_game_band = (
    T_trans
    .join(
        T_date.select("dateid", "datefull"),
        T_date.dateid == T_trans.VisitDateId,
        "left"
    )
    .join(
        T_location.select("LocationID", "Province_CD", "LC_Trade_Brand_Description", "LC_Location_Type_Description"),
        T_location.LocationID == T_trans.LocationId,
        "left"
    )
    .join(
        T_item.select("itemskey", "Item_Description"),
        T_item.itemskey == T_trans.ItemSkey,
        "left"
    )
    .join(bin_mapping_df, on="Item_Description", how="left")
    .filter(F.col("datefull").between("2025-01-01", "2025-12-31"))   # Full year 2025
    .filter(F.col("LC_Location_Type_Description").like("%Restaurant%"))
    .filter(F.col("Item_Description").isin(tags))
)

# ── 4. Summarize total sales by price bin ─────────────────────────────────────

T_gameband_2025_summary = (
    T_game_band
    .groupBy("Province_CD","price_bin")
    .agg(
        F.sum("Net_Revenue").alias("Total_Revenue"),
        F.sum("Quantity").alias("Total_Sales"),
        F.sum(F.when(F.col("SCENEFLAG") == "Yes", 1).otherwise(0)).alias("Scene_Sales_Count"),
        F.sum(F.when(F.col("CineClubFLAG") == "Yes", 1).otherwise(0)).alias("CineClub_Sales_Count"),
        F.countDistinct(F.when(F.col("CDE_ID").isNotNull(), F.col("CDE_ID"))).alias("Distinct_Customers")
    )
    .orderBy("Province_CD","price_bin")
)

T_gameband_2025_summary.display()